# 🤖 Google ADK — Single Agent Hands-On Lab

**For professors & students** | Runs entirely on Google Colab

---

## What you'll build

A single AI agent powered by **Google Gemini** that can:
- Look up weather for any city
- Do maths (add, multiply)
- Save and recall notes within the session
- Tell the current time

You'll see **how the agent decides which tool to use**, inspect the raw event stream, and run a multi-turn conversation.

---

## Before you start

1. **Get a free Gemini API key** → [aistudio.google.com](https://aistudio.google.com)
2. **Add it to Colab Secrets**:
   - Click the 🔑 key icon in the left sidebar
   - Add a secret named `GOOGLE_API_KEY` with your key as the value
   - Toggle **Notebook access** ON
3. Run cells **top to bottom** in order.
4. After Cell 1, you must **restart the runtime** (Runtime → Restart session), then continue from Cell 2.

---
## Cell 1 — Install Google ADK

> ⚠️ After running this cell, go to **Runtime → Restart session**, then continue.

In [ ]:
!pip install -q google-adk
print("✅ google-adk installed. Now restart the runtime: Runtime → Restart session")

---
## Cell 2 — Load API Key

Reads your Gemini API key from Colab Secrets (never hardcode it in the notebook).

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("✅ API key loaded from Colab Secrets.")

---
## Cell 3 — Import ADK

Import the core ADK components we'll use throughout the lab.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext
from google.genai.types import Content, Part

print("✅ ADK imports successful.")
print()
print("Key classes loaded:")
print("  LlmAgent           — the AI agent")
print("  InMemorySessionService — stores conversation history")
print("  Runner             — runs agent turn-by-turn")
print("  ToolContext        — lets tools read/write session state")
print("  Content / Part     — message format for Gemini")

---
## Cell 4 — Define the Tools

Tools are just **plain Python functions**. ADK reads the docstring and type hints automatically to tell the agent what each tool does and what arguments it expects.

> 📌 **Key insight:** The docstring IS the prompt for the tool. Write it clearly — the agent decides when to call a tool based entirely on reading it.

In [ ]:
# ── Tool 1: Weather lookup ─────────────────────────────────────────────────

def get_weather(city: str) -> dict:
    """Get the current weather for a given city.

    Args:
        city: The name of the city to look up (e.g. 'London', 'Tokyo').

    Returns:
        A dict with 'temperature' (Celsius) and 'condition' (str).
    """
    # Mock data — swap this for a real weather API in production
    weather_db = {
        "london":  {"temperature": 15, "condition": "cloudy"},
        "tokyo":   {"temperature": 28, "condition": "sunny"},
        "new york":{"temperature": 22, "condition": "partly cloudy"},
        "paris":   {"temperature": 18, "condition": "rainy"},
        "sydney":  {"temperature": 20, "condition": "clear"},
        "mumbai":  {"temperature": 34, "condition": "humid"},
        "bangalore": {"temperature": 26, "condition": "pleasant"},
    }
    key = city.strip().lower()
    return weather_db.get(key, {"temperature": 20, "condition": "unknown — city not in database"})


# ── Tool 2: Addition ───────────────────────────────────────────────────────

def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return the result.

    Args:
        a: The first number.
        b: The second number.

    Returns:
        The sum of a and b.
    """
    return a + b


# ── Tool 3: Multiplication ─────────────────────────────────────────────────

def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers together and return the result.

    Args:
        a: The first number.
        b: The second number.

    Returns:
        The product of a and b.
    """
    return a * b


# ── Tool 4: Save a note (uses session state) ───────────────────────────────

def save_note(text: str, tool_context: ToolContext) -> str:
    """Save a note to memory for later retrieval.

    Args:
        text: The note content to save.
        tool_context: Injected automatically by ADK — do not pass manually.

    Returns:
        A confirmation message.
    """
    notes = tool_context.state.get("notes", [])
    notes.append(text)
    tool_context.state["notes"] = notes
    return f"Note saved. You now have {len(notes)} note(s) stored."


# ── Tool 5: List saved notes (uses session state) ──────────────────────────

def list_notes(tool_context: ToolContext) -> str:
    """Retrieve all notes saved in this session.

    Args:
        tool_context: Injected automatically by ADK — do not pass manually.

    Returns:
        A formatted string of all saved notes, or a message if none exist.
    """
    notes = tool_context.state.get("notes", [])
    if not notes:
        return "No notes saved yet."
    numbered = "\n".join(f"{i+1}. {note}" for i, note in enumerate(notes))
    return f"Saved notes:\n{numbered}"


# ── Tool 6: Current time ───────────────────────────────────────────────────

def get_current_time() -> str:
    """Get the current date and time.

    Returns:
        The current UTC date and time as a formatted string.
    """
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    return now.strftime("%A, %d %B %Y at %H:%M UTC")


# Show all tools defined
all_tools = [get_weather, add_numbers, multiply_numbers, save_note, list_notes, get_current_time]
print("✅ Tools defined:")
for t in all_tools:
    print(f"   • {t.__name__}() — {t.__doc__.strip().splitlines()[0]}")

---
## Cell 5 — Create the Agent

We wrap our tools in an `LlmAgent`. The four key fields:

| Field | Purpose |
|---|---|
| `name` | Internal identifier |
| `model` | Which Gemini model to use |
| `instruction` | System prompt — guides the agent's behaviour |
| `tools` | List of Python functions the agent can call |

In [ ]:
root_agent = LlmAgent(
    name="smart_assistant",
    model="gemini-2.0-flash",
    description="A helpful assistant with weather, maths, time, and notes tools.",
    instruction=(
        "You are a helpful and concise assistant. "
        "Use your tools whenever a question requires live data or calculation. "
        "For weather, always report both temperature and condition. "
        "For notes, confirm what was saved. "
        "Keep responses brief and friendly."
    ),
    tools=[
        get_weather,
        add_numbers,
        multiply_numbers,
        save_note,
        list_notes,
        get_current_time,
    ],
)

print(f"✅ Agent '{root_agent.name}' created.")
print(f"   Model      : {root_agent.model}")
print(f"   Tools ({len(root_agent.tools)}): {[t.__name__ if hasattr(t,'__name__') else str(t) for t in all_tools]}")

---
## Cell 6 — Set Up the Runner & Session

- **`InMemorySessionService`** — keeps conversation history in RAM (resets when Colab disconnects)
- **`Runner`** — orchestrates the turn-by-turn execution loop
- **`Session`** — a unique conversation context (can have many per app)

In [ ]:
APP_NAME  = "adk_lab"
USER_ID   = "student_01"

session_service = InMemorySessionService()

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

session = session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
)

print("✅ Runner and session ready.")
print(f"   Session ID : {session.id}")
print(f"   App name   : {APP_NAME}")
print(f"   User ID    : {USER_ID}")

---
## Cell 7 — Helper: `ask()` function

This helper sends a message and prints only the final response. We'll use it for clean demos.

Below it we also define `ask_verbose()` which prints **every event** including tool calls — great for teaching how the agent reasons step by step.

In [ ]:
def ask(question: str) -> None:
    """Send a question to the agent and print the final response."""
    print(f"\n{'─'*55}")
    print(f"🧑 User : {question}")
    print(f"{'─'*55}")

    message = Content(role="user", parts=[Part(text=question)])

    for event in runner.run(
        user_id=USER_ID,
        session_id=session.id,
        new_message=message,
    ):
        if event.is_final_response():
            print(f"🤖 Agent: {event.content.parts[0].text.strip()}")


def ask_verbose(question: str) -> None:
    """Send a question and print every event — tool calls AND final response."""
    print(f"\n{'─'*55}")
    print(f"🧑 User : {question}")
    print(f"{'─'*55}")

    message = Content(role="user", parts=[Part(text=question)])

    for event in runner.run(
        user_id=USER_ID,
        session_id=session.id,
        new_message=message,
    ):
        if not event.content:
            continue
        for part in event.content.parts:
            # Tool call — agent is invoking a function
            if hasattr(part, "function_call") and part.function_call:
                fc = part.function_call
                print(f"  🔧 [tool call ] {fc.name}({dict(fc.args)})")
            # Tool result — the function's return value
            elif hasattr(part, "function_response") and part.function_response:
                fr = part.function_response
                print(f"  📦 [tool result] {fr.name} → {fr.response}")
            # Final text response
            elif hasattr(part, "text") and part.text and event.is_final_response():
                print(f"🤖 Agent: {part.text.strip()}")


print("✅ ask() and ask_verbose() helpers ready.")

---
## Cell 8 — Demo: Weather Tool

Ask about the weather. The agent will call `get_weather()` automatically.

In [ ]:
ask("What's the weather like in Tokyo right now?")

---
## Cell 9 — Demo: Maths Tools

Ask a calculation. The agent decides whether to add or multiply.

In [ ]:
ask("What is 47.5 plus 82.3?")

In [ ]:
ask("Multiply 12 by 15 for me.")

---
## Cell 10 — Demo: Notes (Session State)

Notes persist across turns in the same session — the agent can save and retrieve them.

In [ ]:
ask("Please save a note: ADK stands for Agent Development Kit.")

In [ ]:
ask("Save another note: Gemini is Google's family of AI models.")

In [ ]:
ask("What notes have I saved?")

---
## Cell 11 — Demo: Multi-Tool Question

A single question that requires multiple tool calls. Watch the agent chain them together.

In [ ]:
ask("What is the weather in London and Paris? Also what time is it now?")

---
## Cell 12 — Verbose Mode: See the Agent's Reasoning

Use `ask_verbose()` to see every tool call and result before the final answer.

> 📌 This is the key teaching cell — students can see **exactly how the agent decides** which tool to invoke and what the tool returns.

In [ ]:
ask_verbose("What is the weather in Bangalore? And multiply 9 by 7.")

---
## Cell 13 — Multi-turn Conversation

All previous turns are remembered in the session. The agent can refer back to earlier messages.

In [ ]:
ask("What is 10 plus 5?")

In [ ]:
ask("Now multiply that result by 3.")  # refers to previous answer: 15 × 3

In [ ]:
ask("Save that final number as a note.")

---
## Cell 14 — Inspect Session State Directly

You can read the session state at any time to see what's been stored.

In [ ]:
# Retrieve the current session object and inspect its state
current_session = session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=session.id,
)

print("📦 Session state contents:")
print(current_session.state)

print("\n💬 Conversation turns so far:", len(current_session.events))

---
## Cell 15 — Exercise: Add Your Own Tool 🏗️

**Your turn!** Add a new tool to the agent.

Starter: a `unit_converter` tool that converts km → miles (or any conversion you like).

Steps:
1. Define the function with a clear docstring and type hints
2. Add it to the `tools=` list when creating a new agent
3. Test it with a natural language question

In [ ]:
# ── Your new tool ──────────────────────────────────────────────────────────

def km_to_miles(km: float) -> float:
    """Convert a distance from kilometres to miles.

    Args:
        km: Distance in kilometres.

    Returns:
        Equivalent distance in miles (rounded to 2 decimal places).
    """
    return round(km * 0.621371, 2)


# ── Rebuild agent with the new tool ──────────────────────────────────────

extended_agent = LlmAgent(
    name="extended_assistant",
    model="gemini-2.0-flash",
    description="Assistant with weather, maths, time, notes, and unit conversion.",
    instruction=(
        "You are a helpful assistant. Use tools when needed. "
        "For unit conversions, always state both the original and converted value."
    ),
    tools=[
        get_weather,
        add_numbers,
        multiply_numbers,
        save_note,
        list_notes,
        get_current_time,
        km_to_miles,      # ← new tool added here
    ],
)

# New runner + session for this agent
session_service_ex = InMemorySessionService()
runner_ex = Runner(
    agent=extended_agent,
    app_name="extended_lab",
    session_service=session_service_ex,
)
session_ex = session_service_ex.create_session(
    app_name="extended_lab", user_id="student_01"
)

def ask_ex(question: str) -> None:
    """Ask the extended agent."""
    print(f"\n{'─'*55}")
    print(f"🧑 User : {question}")
    print(f"{'─'*55}")
    message = Content(role="user", parts=[Part(text=question)])
    for event in runner_ex.run(
        user_id="student_01",
        session_id=session_ex.id,
        new_message=message,
    ):
        if event.is_final_response():
            print(f"🤖 Agent: {event.content.parts[0].text.strip()}")


# ── Test your new tool ───────────────────────────────────────────────────
ask_ex("How many miles is 100 kilometres?")

---
## Cell 16 — Challenge Questions 🎯

Try these in any order. Uncomment and run one at a time.

In [ ]:
# Challenge 1: Ask a question where the agent uses NO tool (answers from knowledge)
ask("What does ADK stand for?")

In [ ]:
# Challenge 2: Ask a question that requires chaining add then multiply
ask("Add 20 and 30, then multiply that result by 4.")

In [ ]:
# Challenge 3: Ask in a natural, ambiguous way — does the agent pick the right tool?
ask("Is it warm in Sydney today?")

In [ ]:
# Challenge 4: Try to confuse the agent — ask for a city not in the database
ask("What's the weather in Timbuktu?")

In [ ]:
# Challenge 5 (modify this): Add a new tool of your own design
#   Ideas: currency converter, BMI calculator, days between dates
#   Then add it to extended_agent's tools list and test it!

# def my_new_tool(...):
#     """Description here."""
#     pass

---
## Summary

You've built a complete single-agent system with Google ADK. Here's what each part does:

| Component | What you wrote | What ADK does with it |
|---|---|---|
| **Tool function** | Plain Python function + docstring | Generates tool schema for Gemini |
| **`LlmAgent`** | `name`, `model`, `instruction`, `tools` | Routes LLM reasoning + tool calls |
| **`InMemorySessionService`** | — | Stores conversation history in RAM |
| **`Runner`** | — | Manages the turn-by-turn loop |
| **`ToolContext`** | Used inside tool functions | Reads/writes session `state` dict |
| **Event loop** | `for event in runner.run(...)` | Exposes every step including tool calls |

---

## Next steps

- **Multi-agent**: Wrap agents as `AgentTool` objects and build an orchestrator
- **Real APIs**: Replace mock weather data with a live OpenWeatherMap API call
- **Persistent state**: Swap `InMemorySessionService` for a database-backed service
- **Streaming**: Display responses token-by-token using the streaming event API

---
*Google ADK docs: https://google.github.io/adk-docs/*